In [3]:
# Zadanie 5 - N-gramy w TF-IDF

"""Porownaj wyniki klasyfikacji na IMDB z roznymi ngram_range:
(1,1) -- tylko unigramy
(1,2) -- unigramy + bigramy
(1,3) -- unigramy + bigramy + trigramy
(2,2) -- tylko bigramy

Zmierz accuracy i rozmiar macierzy TF-IDF.

Oczekiwany wynik: tabela wynikow, analiza
"""

import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# =========================
# Wczytaj IMDB jako tekst
# =========================
max_features = 10000
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=max_features)

word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {v: k for k, v in word_index.items()}

def decode_review(sequence):
    return " ".join([reverse_word_index.get(i - 3, "?") for i in sequence])

x_train_text = [decode_review(seq) for seq in x_train]
x_test_text = [decode_review(seq) for seq in x_test]

# =========================
# Konfiguracje n-gramów
# =========================
configs = {
    "(1,1)": (1,1),
    "(1,2)": (1,2),
    "(1,3)": (1,3),
    "(2,2)": (2,2),
}

results = []

for name, ngram_range in configs.items():

    vectorizer = TfidfVectorizer(
        max_features=20000,
        ngram_range=ngram_range
    )

    X_train = vectorizer.fit_transform(x_train_text)
    X_test = vectorizer.transform(x_test_text)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    results.append({
        "ngram_range": name,
        "accuracy": acc,
        "num_features": X_train.shape[1]
    })

df = pd.DataFrame(results)
print(df)

/home/wojtek/PycharmProjects/my_first_ds_project/.venv/lib/python3.12/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


  ngram_range  accuracy  num_features
0       (1,1)   0.88416          9771
1       (1,2)   0.89532         20000
2       (1,3)   0.89460         20000
3       (2,2)   0.87524         20000


In [5]:
# Zadanie 10 - Pretrenowane embeddingi w LSTM
"""
Porownaj LSTM z embeddingami:
Trenowanymi od zera
Pretrenowanymi GloVe (zamrozonymi, trainable=False)
Pretrenowanymi GloVe (odblokowanymi, trainable=True)

Dataset: IMDB.

Oczekiwany wynik: porownanie 3 wariantow, analiza
"""

import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import time

# =========================
# Parametry
# =========================
max_words = 10000
maxlen = 200
embedding_dim = 100

# =========================
# Dane IMDB (tekst)
# =========================
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=max_words)

word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {v: k for k, v in word_index.items()}

def decode(seq):
    return " ".join([reverse_word_index.get(i - 3, "?") for i in seq])

x_train_text = [decode(s) for s in x_train]
x_test_text = [decode(s) for s in x_test]

# Tokenizer
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(x_train_text)

X_train = tokenizer.texts_to_sequences(x_train_text)
X_test = tokenizer.texts_to_sequences(x_test_text)

X_train = pad_sequences(X_train, maxlen=maxlen)
X_test = pad_sequences(X_test, maxlen=maxlen)

# =========================
# Wczytaj GloVe
# =========================
embeddings_index = {}
with open("glove.6B.100d.txt") as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

# macierz embeddingów
embedding_matrix = np.zeros((max_words, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i < max_words:
        vec = embeddings_index.get(word)
        if vec is not None:
            embedding_matrix[i] = vec

# =========================
# Funkcja modelu
# =========================
def build_model(trainable_embedding, use_glove):
    if use_glove:
        embedding_layer = layers.Embedding(
            max_words,
            embedding_dim,
            weights=[embedding_matrix],
            trainable=trainable_embedding
        )
    else:
        embedding_layer = layers.Embedding(max_words, embedding_dim)

    model = keras.Sequential([
        embedding_layer,
        layers.LSTM(64),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

# =========================
# 3 warianty
# =========================
configs = [
    ("scratch", False, True),
    ("glove_frozen", True, False),
    ("glove_trainable", True, True)
]

results = []

for name, use_glove, trainable in configs:

    model = build_model(trainable, use_glove)

    start = time.time()

    history = model.fit(
        X_train, y_train,
        epochs=3,
        batch_size=64,
        validation_split=0.2,
        verbose=0
    )

    end = time.time()

    acc = model.evaluate(X_test, y_test, verbose=0)[1]

    results.append({
        "model": name,
        "accuracy": acc,
        "time": end - start
    })

import pandas as pd
df = pd.DataFrame(results)
print(df)

E0000 00:00:1776954356.923820   13885 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1776954356.950587   13885 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


             model  accuracy       time
0          scratch   0.86312  55.161500
1     glove_frozen   0.83892  48.158127
2  glove_trainable   0.87920  55.911362
